In [1]:
#import required libraries and specify paths for atttack and benign CSV and directory to store ttrain and test CSVs

import pandas as pd
import numpy as np
import os
import collections
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler 

print("Imported all required libraries")

attack_csv=r"C:\Users\NARASIMHA N\OneDrive\Desktop\Kitsune Feature Extraction\Dataset Preprocessing\Remove Uncommon Features\UQIoT_Attack.csv"
benign_csv=r"C:\Users\NARASIMHA N\OneDrive\Desktop\Kitsune Feature Extraction\Dataset Preprocessing\Remove Uncommon Features\UQIoT_Benign.csv"

output_dir=r"C:\Users\NARASIMHA N\OneDrive\Desktop\Kitsune Feature Extraction\Dataset Preprocessing\UQ-IoT Dataset\Testing and Training CSVs"

print("Paths are set.")

Imported all required libraries
Paths are set.


In [2]:
#load CSV files

attack_df=pd.read_csv(attack_csv)
benign_df=pd.read_csv(benign_csv)

print(f"Attack Dataset: {attack_df.shape}")
print(f"Benign Dataset: {benign_df.shape}")

Attack Dataset: (15302, 104)
Benign Dataset: (20000, 104)


In [3]:
#Encode labels

print("Before Encoding:")
print(f"Attack label values: {attack_df['label'].unique()}")
print(f"Benign label values: {benign_df['label'].unique()}")

attack_df['label']=attack_df['label'].map({'ARP_Spoofing':1})
benign_df['label']=benign_df['label'].map({'Benign':0})

print("After Encoding:")
print(f"Attack label values: {attack_df['label'].unique()}")
print(f"Benign label values: {benign_df['label'].unique()}")

print(f"NaN in attack labels: {attack_df['label'].isna().sum()}")
print(f"NaN in Benign labels: {benign_df['label'].isna().sum()}")

Before Encoding:
Attack label values: <StringArray>
['ARP_Spoofing']
Length: 1, dtype: str
Benign label values: <StringArray>
['Benign']
Length: 1, dtype: str
After Encoding:
Attack label values: [1]
Benign label values: [0]
NaN in attack labels: 0
NaN in Benign labels: 0


In [4]:
#Split each class separately using train-test-split

attack_train, attack_test=train_test_split(attack_df, test_size=0.2, random_state=42)

benign_train, benign_test=train_test_split(benign_df, test_size=0.2, random_state=42)

print(f"Attack - Train: {len(attack_train)} / Test: {len(attack_test)}")
print(f"Benign - Train: {len(benign_train)} / Test: {len(benign_test)}")


Attack - Train: 12241 / Test: 3061
Benign - Train: 16000 / Test: 4000


In [5]:
#Build test set by combining attack nad benign test rows and shuffel them

test_df=pd.concat([attack_test, benign_test]).sample(frac=1.0, random_state=42).reset_index(drop=True)

ratio=len(benign_test)/len(attack_test)

print(f"Test set size: {len(test_df)}")
print(f"Attack Rows in test set: {len(attack_test)}")
print(f"Benign Rows in test set: {len(benign_test)}")
print(f"Attack to Benign ratio: {round(ratio,2),":1"}")
print(f"Class distribution: {collections.Counter(test_df['label'])}")

Test set size: 7061
Attack Rows in test set: 3061
Benign Rows in test set: 4000
Attack to Benign ratio: (1.31, ':1')
Class distribution: Counter({0: 4000, 1: 3061})


In [6]:
#Build Train Dataset by combining attack and benign train rows

train_df=pd.concat([attack_train,benign_train]).sample(frac=1.0, random_state=42).reset_index(drop=True)

#Find all feature columns except label column
feature_columns = [feature for feature in attack_df.columns if feature != 'label']

X_train=train_df[feature_columns]
y_train=train_df['label']

print(f"Before Undersampling: {collections.Counter(y_train)}")

reduce=RandomUnderSampler(sampling_strategy=1.0, random_state=42)
X_train_resampled, y_train_resampled=reduce.fit_resample(X_train, y_train)

print("\nAfter Undersampling:")
print(collections.Counter(y_train_resampled))

Before Undersampling: Counter({0: 16000, 1: 12241})

After Undersampling:
Counter({0: 12241, 1: 12241})


In [7]:
#Apply standard Scaler 

scaler=StandardScaler()

X_train_scaled=scaler.fit_transform(X_train_resampled)

X_test_scaled=scaler.transform(test_df[feature_columns])

print(f"Train shape after scaling: {X_train_scaled.shape}")
print(f"Test shape after scaling: {X_test_scaled.shape}")

Train shape after scaling: (24482, 103)
Test shape after scaling: (7061, 103)


In [8]:
#Save Train and Test CSVs

train_csv=pd.DataFrame(X_train_scaled, columns=feature_columns)
train_csv['label']=y_train_resampled.values

test_csv=pd.DataFrame(X_test_scaled, columns=feature_columns)
test_csv['label']=test_df['label'].values

train_path=os.path.join(output_dir, "UQ-IoT_train.csv")
test_path=os.path.join(output_dir, "UQ-IoT_test.csv")

train_csv.to_csv(train_path, index=False)
test_csv.to_csv(test_path, index=False)

print(f"Train CSV saved to: {train_path}")
print(f"Test CSV save to: {test_path}")
print(f"Train shape: {train_csv.shape}")
print(f"Test shape: {test_csv.shape}")

Train CSV saved to: C:\Users\NARASIMHA N\OneDrive\Desktop\Kitsune Feature Extraction\Dataset Preprocessing\UQ-IoT Dataset\Testing and Training CSVs\UQ-IoT_train.csv
Test CSV save to: C:\Users\NARASIMHA N\OneDrive\Desktop\Kitsune Feature Extraction\Dataset Preprocessing\UQ-IoT Dataset\Testing and Training CSVs\UQ-IoT_test.csv
Train shape: (24482, 104)
Test shape: (7061, 104)
